# Climate Report Analyzer using ClimateBERT

A comprehensive pipeline for extracting, translating, and analyzing climate-related content from corporate sustainability reports (PDFs).

## 🔄 Pipeline Overview

1. **Extract** → Text extraction from PDFs (handles multi-column layouts, excludes tables)
2. **Translate** → Non-English reports → English (Helsinki-NLP models)
3. **Filter** → Identify climate-relevant chunks (ClimateBERT detector)
4. **Analyze** → Score specificity, commitment, sentiment, and net-zero focus

## ✨ Features

- ✅ Caching system (JSON) to avoid reprocessing
- ✅ GPU acceleration with automatic memory management
- ✅ Batch processing with progress tracking
- ✅ Supports 11 languages via Helsinki-NLP translation models

## 🖥️ Hardware Requirements

**Optimized for NVIDIA GPUs with 4GB+ VRAM** (tested on T1200)
- Falls back to CPU if no GPU available
- GPU memory management handles OOM gracefully
- Batch sizes auto-adjust based on available memory

## 📦 Outputs

- `*_prep.json`: Extracted and translated chunks
- `*_bert.json`: Climate analysis results with all ClimateBERT scores

---

# 1. Verify GPU detection

In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("✅ CUDA GPU detected!")
elif torch.backends.mps.is_available():
    print("✅ Apple Silicon (MPS) detected!")
else:
    print("⚠️ No GPU detected - will use CPU (slower)")

# ❌ Linux-only troubleshooting commands (keep commented)
# sudo nvidia-smi
# sudo nvidia-smi -pm 1 # wake up if sleepy/off
# echo $LD_LIBRARY_PATH # test cuda in path
# echo $PATH | grep cuda
# sudo rmmod nvidia_uvm #nvidia_drm, nvidia_modeset, nvidia # Unload nvidia module
# sudo modprobe nvidia_uvm # Reload them

PyTorch version: 2.9.1+cu128
CUDA version: 12.8
CUDA available: True
GPU: NVIDIA T1200 Laptop GPU
✅ CUDA GPU detected!


In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("✅ SUCCESS!")
else:
    print("⚠️ No GPU detected - will use CPU (slower)")


# ❌ Linux-only troubleshooting commands
# ---------------------------------------------------------


PyTorch version: 2.9.1+cu130
CUDA version: 13.0
CUDA available: True
GPU: NVIDIA T1200 Laptop GPU
✅ SUCCESS!


## 2. Run Pipeline

In [3]:
from transformers import MarianMTModel, MarianTokenizer, pipeline
from tqdm.auto import tqdm
import fitz  # PyMuPDF
import langid
import spacy
import torch
import os
import gc
import re
import json
import time
import logging
import unicodedata
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
from collections import Counter

# === CRITICAL: Must be set BEFORE importing torch ===
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.7,max_split_size_mb:256"


# Suppress verbose transformer warnings
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# Load spacy once
_nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "tagger"])
_nlp.max_length = 2_000_000


class ClimateReportAnalyzer:
    """
    Processes PDF reports through ClimateBERT pipeline.

    Pipeline: Extract → Translate → Filter → Analyze
    Output: JSON files with scored climate chunks

    Args:
        cache_dir: Directory for cached results (default: "cache")
        use_torch_compile: Use torch.compile for faster inference (default: True)
        verbose: Print detailed per-file output (default: False)
    """

    # === SETTINGS ===
    MIN_CHARS = 600
    MAX_CHARS = 1600
    MAX_TOKENS = MAX_CHARS // 4
    BATCH_SIZE = 32

    TRANSLATE_BATCH_SIZE = 32
    TRANSLATE_INPUT_MAX_TOKENS = 512
    TRANSLATE_OUTPUT_MAX_TOKENS = 512

    HELSINKI_MODELS = {
        'de': 'Helsinki-NLP/opus-mt-de-en',
        'es': 'Helsinki-NLP/opus-mt-es-en',
        'it': 'Helsinki-NLP/opus-mt-it-en',
        'zh': 'Helsinki-NLP/opus-mt-zh-en',
        'fr': 'Helsinki-NLP/opus-mt-fr-en',
        'nl': 'Helsinki-NLP/opus-mt-nl-en',
        'pt': 'Helsinki-NLP/opus-mt-ROMANCE-en',
        'pl': 'Helsinki-NLP/opus-mt-pl-en',
        'ru': 'Helsinki-NLP/opus-mt-ru-en',
        'ja': 'Helsinki-NLP/opus-mt-ja-en',
        'ko': 'Helsinki-NLP/opus-mt-ko-en',
    }

    SPAM_CHARS = set('•·*─―–-')

    def __init__(self, cache_dir="cache", use_torch_compile=True, verbose=False):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        self.pdf_path = None
        self.use_torch_compile = use_torch_compile
        self.verbose = verbose

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translator = None
        self.translator_lang = None
        self.prep_data = None
        self.bert_data = None

        # Report GPU status once
        if self.device >= 0:
            gpu_name = torch.cuda.get_device_name(0)
            total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"✓ GPU: {gpu_name} ({total_mem:.1f}GB)")
            self._clear_gpu_memory()
        else:
            print("✓ Running on CPU (no GPU detected)")

    # === LOGGING HELPERS ===

    def _log(self, msg):
        """Print only if verbose mode enabled."""
        if self.verbose:
            print(msg)

    # === GPU MEMORY MANAGEMENT ===

    def _get_gpu_memory_info(self) -> Dict:
        if self.device < 0:
            return {'available': False}
        return {
            'available': True,
            'allocated_gb': torch.cuda.memory_allocated(0) / 1e9,
            'reserved_gb': torch.cuda.memory_reserved(0) / 1e9,
            'total_gb': torch.cuda.get_device_properties(0).total_memory / 1e9,
            'free_gb': (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
        }

    def _clear_gpu_memory(self):
        if self.device >= 0:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    def _emergency_gpu_cleanup(self):
        if self.device < 0:
            return

        # Unload translator
        if self.translator:
            try:
                if 'model' in self.translator:
                    self.translator['model'].cpu()
                    del self.translator['model']
                if 'tokenizer' in self.translator:
                    del self.translator['tokenizer']
            except:
                pass
            self.translator = None
            self.translator_lang = None

        # Unload all models
        for name in list(self.models.keys()):
            try:
                self.models[name].model.cpu()
                del self.models[name]
            except:
                pass
        self.models = {}

        gc.collect()

        # Move any remaining CUDA tensors to CPU
        for obj in gc.get_objects():
            try:
                if torch.is_tensor(obj) and obj.is_cuda:
                    obj.data = obj.data.cpu()
            except:
                pass

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    def _unload_translator(self):
        if self.translator:
            self._log("  Unloading translator...")
            try:
                if 'model' in self.translator:
                    self.translator['model'].cpu()
                    del self.translator['model']
                if 'tokenizer' in self.translator:
                    del self.translator['tokenizer']
            except:
                pass
            self.translator = None
            self.translator_lang = None
            self._clear_gpu_memory()

    # === TRANSLATION ===

    def _load_helsinki_translator(self, src_lang: str):
        if self.translator and self.translator_lang == src_lang:
            return self.translator
        if self.translator:
            self._unload_translator()

        model_name = self.HELSINKI_MODELS.get(src_lang)
        if not model_name:
            return None

        self._log(f"  Loading translator: {model_name}")

        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device >= 0 else torch.float32,
        )

        if self.device >= 0:
            model = model.cuda()
            if self.use_torch_compile and hasattr(torch, 'compile'):
                try:
                    model = torch.compile(model, mode="reduce-overhead")
                except:
                    pass
        model.eval()

        self.translator = {'model': model,
                           'tokenizer': tokenizer, 'type': 'helsinki'}
        self.translator_lang = src_lang
        return self.translator

    def _translate_chunks_helsinki(self, chunks: List[str], src_lang: str) -> List[str]:
        translator = self._load_helsinki_translator(src_lang)
        model = translator['model']
        tokenizer = translator['tokenizer']

        translated = []
        batch_size = max(8, self.TRANSLATE_BATCH_SIZE // 2)

        # No tqdm here - controlled by outer loop
        with torch.no_grad():
            for i in range(0, len(chunks), batch_size):
                batch = chunks[i:i + batch_size]
                try:
                    inputs = tokenizer(batch, return_tensors="pt", padding=True,
                                       truncation=True, max_length=self.TRANSLATE_INPUT_MAX_TOKENS)
                    if self.device >= 0:
                        inputs = {k: v.cuda() for k, v in inputs.items()}

                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                        no_repeat_ngram_size=3,
                        repetition_penalty=1.2,
                        num_beams=4,
                        early_stopping=True,
                        do_sample=False,
                        use_cache=True,
                    )
                    batch_translations = tokenizer.batch_decode(
                        outputs, skip_special_tokens=True)
                    translated.extend(batch_translations)
                    del inputs, outputs

                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                        # Fallback to single-item processing
                        for chunk in batch:
                            try:
                                inputs = tokenizer([chunk], return_tensors="pt",
                                                   truncation=True, max_length=self.TRANSLATE_INPUT_MAX_TOKENS)
                                if self.device >= 0:
                                    inputs = {k: v.cuda()
                                              for k, v in inputs.items()}
                                output = model.generate(**inputs, max_new_tokens=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                                                        no_repeat_ngram_size=3, num_beams=1)
                                translated.append(tokenizer.decode(
                                    output[0], skip_special_tokens=True))
                                del inputs, output
                            except:
                                translated.append(chunk)
                                torch.cuda.empty_cache()
                    else:
                        raise

        # Post-processing: clean and filter repetitions
        cleaned = []
        for trans in translated:
            trans = self._clean_artifacts(trans)
            if not self._detect_severe_repetition(trans):
                cleaned.append(trans)
            else:
                trans = self._remove_repetitions(trans, max_repeat=2)
                if not self._detect_severe_repetition(trans):
                    cleaned.append(trans)

        return cleaned

    # === TEXT PROCESSING ===

    def _fix_pdf_encoding(self, text: str) -> str:
        text = unicodedata.normalize('NFC', text)
        text = re.sub(r'\(cid:\d+\)', '', text)
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
        for char in ['\u00ad', '\u200b', '\u200c', '\u200d', '\ufeff']:
            text = text.replace(char, '')
        return text

    def _is_noise_line(self, line: str) -> bool:
        line = line.strip()
        if len(line) < 3:
            return True
        words = line.split()
        if len(words) > 5:
            single_chars = sum(1 for w in words if len(w) <= 2)
            if single_chars / len(words) > 0.7:
                return True
        if re.match(r'^.{5,50}\.{5,}\s*\d+$', line):
            return True
        if re.match(r'^(page|p\.?)\s*\d+|^\d+\s*(of|/)\s*\d+$', line, re.I):
            return True
        if re.match(r'^[\d\s\.\-\/]+$', line) and len(line) < 15:
            return True
        if len(words) > 8:
            num_count = sum(1 for w in words if re.match(
                r'^\d+[\.\,]?\d*$', w))
            if num_count / len(words) > 0.6:
                return True
        if re.match(r'^https?://\S+$', line, re.I):
            return True
        spam_count = sum(1 for c in line if c in self.SPAM_CHARS)
        if spam_count > 5:
            return True
        if len(line) > 10 and line.count('.') / len(line) > 0.4:
            return True
        if re.match(r'^[A-Z][a-z]+(\s+[A-Z][a-z]+){10,}$', line):
            return True
        return False

    def _clean_artifacts(self, text: str) -> str:
        text = re.sub(
            r'\b([A-ZÄÖÜ])\s+([A-ZÄÖÜ])\s+([A-ZÄÖÜ])', r'\1\2\3', text)
        text = re.sub(r'\.{5,}', '...', text)
        text = re.sub(r'([.\-–—•·])\1{3,}', r'\1\1', text)
        text = re.sub(r'\s{3,}', ' ', text)
        return text.strip()

    def _detect_severe_repetition(self, text: str) -> bool:
        words = text.split()
        if len(words) < 10:
            return False
        for i in range(len(words) - 4):
            if words[i] == words[i+1] == words[i+2] == words[i+3] == words[i+4]:
                return True
        stopwords = {'the', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'for',
                     'on', 'with', 'is', 'are', 'was', 'were', 'be', 'been'}
        word_counts = Counter(w.lower()
                              for w in words if w.lower() not in stopwords)
        if not word_counts:
            return False
        most_common_word, most_common_count = word_counts.most_common(1)[0]
        content_words = sum(word_counts.values())
        return content_words > 0 and (most_common_count / content_words) > 0.3

    def _remove_repetitions(self, text: str, max_repeat: int = 2) -> str:
        for char in self.SPAM_CHARS:
            pattern = rf'(\s*{re.escape(char)}\s*){{3,}}'
            text = re.sub(pattern, f' {char} ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        words = text.split()
        if len(words) < 4:
            return text
        result = []
        i = 0
        while i < len(words):
            word = words[i]
            count = 1
            while i + count < len(words) and words[i + count] == word:
                count += 1
            result.extend([word] * min(count, max_repeat))
            i += count
        text = ' '.join(result)
        words = text.split()
        for n in [4, 3, 2]:
            if len(words) < n * 2:
                continue
            cleaned_words = []
            i = 0
            while i < len(words):
                if i + n * 2 <= len(words):
                    ngram = ' '.join(words[i:i+n])
                    next_ngram = ' '.join(words[i+n:i+n*2])
                    if ngram == next_ngram:
                        cleaned_words.extend(words[i:i+n])
                        j = i + n
                        while j + n <= len(words) and ' '.join(words[j:j+n]) == ngram:
                            j += n
                        i = j
                    else:
                        cleaned_words.append(words[i])
                        i += 1
                else:
                    cleaned_words.append(words[i])
                    i += 1
            words = cleaned_words
        return ' '.join(words)

    def _prep_text(self, raw_text: str) -> str:
        text = self._fix_pdf_encoding(raw_text)
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
        lines = []
        for line in text.split('\n'):
            line = re.sub(r'[ \t]+', ' ', line).strip()
            if not line:
                lines.append('')
            elif self._is_noise_line(line):
                continue
            else:
                lines.append(line)

        paragraphs = []
        current = []
        for i, line in enumerate(lines):
            if line:
                current.append(line)
                ends_sentence = re.search(r'[.!?]\s*$', line)
                if ends_sentence and (i + 1 >= len(lines) or not lines[i + 1]):
                    para = ' '.join(current)
                    para = self._clean_artifacts(para)
                    if len(para) > 30 and not self._detect_severe_repetition(para):
                        paragraphs.append(para)
                    current = []
            elif current:
                if current and re.search(r'[.!?]\s*$', current[-1]):
                    para = ' '.join(current)
                    para = self._clean_artifacts(para)
                    if len(para) > 30 and not self._detect_severe_repetition(para):
                        paragraphs.append(para)
                    current = []
        if current:
            para = ' '.join(current)
            para = self._clean_artifacts(para)
            if len(para) > 30 and not self._detect_severe_repetition(para):
                paragraphs.append(para)

        return '\n\n'.join(paragraphs)

    def _split_sentences(self, text: str) -> List[str]:
        global _nlp
        doc = _nlp(text)
        return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    def _chunk_text(self, text: str) -> List[str]:
        paragraphs = re.split(r'\n\s*\n', text)
        paragraphs = [p.strip() for p in paragraphs if p.strip()]
        if not paragraphs:
            return []

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_len = len(para)

            if self.MIN_CHARS <= para_len <= self.MAX_CHARS:
                if current_chunk:
                    merged = current_chunk + " " + para
                    if len(merged) <= self.MAX_CHARS:
                        current_chunk = merged
                    else:
                        chunks.append(current_chunk)
                        current_chunk = para
                else:
                    current_chunk = para

            elif para_len > self.MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk)
                    current_chunk = ""
                sentences = self._split_sentences(para)
                temp_chunk = ""
                for sent in sentences:
                    if not temp_chunk:
                        temp_chunk = sent
                    elif len(temp_chunk) + len(sent) + 1 <= self.MAX_CHARS:
                        temp_chunk += " " + sent
                    else:
                        if len(temp_chunk) >= self.MIN_CHARS:
                            chunks.append(temp_chunk)
                        temp_chunk = sent
                if temp_chunk:
                    if len(temp_chunk) >= self.MIN_CHARS:
                        chunks.append(temp_chunk)
                    else:
                        current_chunk = temp_chunk
            else:
                if current_chunk:
                    current_chunk += " " + para
                else:
                    current_chunk = para

        if current_chunk and len(current_chunk) >= self.MIN_CHARS:
            chunks.append(current_chunk)

        return chunks

    def _bbox_overlap(self, bbox1, bbox2, threshold=0.5):
        x1_min, y1_min, x1_max, y1_max = bbox1
        x2_min, y2_min, x2_max, y2_max = bbox2
        x_overlap = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
        y_overlap = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
        if x_overlap == 0 or y_overlap == 0:
            return False
        intersection_area = x_overlap * y_overlap
        bbox1_area = (x1_max - x1_min) * (y1_max - y1_min)
        return (intersection_area / bbox1_area) > threshold if bbox1_area > 0 else False

    def _extract_text_pymupdf(self) -> str:
        all_text = []
        try:
            doc = fitz.open(self.pdf_path)
        except Exception as e:
            self._log(f"  ⚠ PyMuPDF failed: {e}")
            return ""

        for page in doc:
            table_bboxes = []
            try:
                tables = page.find_tables()
                if tables:
                    table_bboxes = [table.bbox for table in tables]
            except:
                pass

            try:
                blocks = page.get_text("dict", sort=True)["blocks"]
            except:
                text = page.get_text("text")
                if text:
                    all_text.append(text)
                continue

            page_paragraphs = []
            for block in blocks:
                if block.get("type") != 0:
                    continue
                bbox = block.get("bbox", [0, 0, 0, 0])
                block_height = bbox[3] - bbox[1]
                if block_height < 5:
                    continue
                is_in_table = any(self._bbox_overlap(
                    bbox, tb, threshold=0.5) for tb in table_bboxes)
                if is_in_table:
                    continue

                block_lines = []
                for line in block.get("lines", []):
                    spans_text = [span.get("text", "").strip() for span in line.get(
                        "spans", []) if span.get("text", "").strip()]
                    if spans_text:
                        block_lines.append(" ".join(spans_text))

                if block_lines:
                    paragraph = " ".join(block_lines)
                    paragraph = re.sub(r'\s+', ' ', paragraph).strip()
                    if paragraph and len(paragraph) > 2:
                        page_paragraphs.append(paragraph)

            if page_paragraphs:
                all_text.append("\n\n".join(page_paragraphs))

        doc.close()
        return "\n\n".join(all_text)

    # === METADATA & CACHING ===

    def _extract_metadata_from_path(self, pdf_path: Path) -> dict:
        pdf_path = Path(pdf_path)
        filename = pdf_path.stem
        parts = pdf_path.parts

        company = None
        for i, part in enumerate(parts):
            if part.lower() == 'reports' and i + 1 < len(parts):
                company = parts[i + 1]
                break
        if not company:
            skip_folders = {'factsheets', 'highlights',
                            'annual', 'reports', 'data', 'pdfs'}
            for parent in [pdf_path.parent, pdf_path.parent.parent]:
                if parent.name.lower() not in skip_folders and parent.name:
                    company = parent.name
                    break

        company_id = None
        match = re.match(r'^(\d{2,3})_', filename)
        if match:
            company_id = match.group(1)

        year = None
        year_matches = re.findall(r'(20[12]\d)', filename)
        if year_matches:
            year = int(year_matches[0])

        return {'company': company, 'company_id': company_id, 'year': year}

    def set_pdf_path(self, pdf_path: str) -> None:
        new_path = Path(pdf_path)
        if self.pdf_path != new_path:
            self.pdf_path = new_path
            self.prep_data = None
            self.bert_data = None

    def _get_cache_path(self, suffix: str) -> Path:
        if not self.pdf_path:
            raise ValueError("No PDF path set")
        return self.cache_dir / f"{self.pdf_path.stem}_{suffix}.json"

    def _load_prep(self) -> Optional[Dict]:
        if self.prep_data is not None:
            return self.prep_data
        cache_file = self._get_cache_path('prep')
        if cache_file.exists():
            self._log(f"  Loading cached prep: {cache_file.name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                self.prep_data = json.load(f)
            return self.prep_data
        return None

    def _save_prep(self) -> None:
        if self.prep_data is None:
            return
        cache_file = self._get_cache_path('prep')
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.prep_data, f, ensure_ascii=False, indent=2)

    def _load_bert(self) -> Optional[Dict]:
        if self.bert_data is not None:
            return self.bert_data
        cache_file = self._get_cache_path('bert')
        if cache_file.exists():
            self._log(f"  Loading cached BERT: {cache_file.name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                self.bert_data = json.load(f)
            return self.bert_data
        return None

    def _save_bert(self) -> None:
        if self.bert_data is None:
            return
        cache_file = self._get_cache_path('bert')
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.bert_data, f, ensure_ascii=False, indent=2)

    def _detect_language(self, text: str) -> str:
        try:
            text_len = len(text)
            samples = []
            if text_len > 15000:
                samples = [text[1000:4000], text[text_len //
                                                 2:text_len//2+3000], text[-4000:-1000]]
            else:
                samples = [text[100:min(5000, text_len-100)]]
            votes = []
            for sample in samples:
                sample_clean = re.sub(r'[^\w\s]', ' ', sample)
                sample_clean = re.sub(r'\s+', ' ', sample_clean)
                if len(sample_clean) > 100:
                    lang, _ = langid.classify(sample_clean)
                    votes.append(lang)
            if votes:
                return Counter(votes).most_common(1)[0][0]
            return 'en'
        except:
            return 'en'

    # === PIPELINE STEPS ===

    def extract_pdf(self) -> Dict:
        if self.prep_data is not None:
            return self.prep_data
        cached = self._load_prep()
        if cached:
            return cached

        metadata = self._extract_metadata_from_path(self.pdf_path)
        raw_text = self._extract_text_pymupdf()
        text = self._prep_text(raw_text)
        lang = self._detect_language(text)
        chunks = self._chunk_text(text)

        chunk_ids = [f"{metadata['company_id']}_{idx:03d}" if metadata['company_id']
                     else f"chunk_{idx:03d}" for idx in range(len(chunks))]

        self.prep_data = {
            "pdf_path": str(self.pdf_path), "company": metadata['company'],
            "company_id": metadata['company_id'], "year": metadata['year'],
            "language": lang, "extraction_method": "pymupdf_blocks",
            "num_chunks": len(chunks), "chunks": chunks, "chunk_ids": chunk_ids,
            "translated": False, "extracted_at": str(datetime.now())
        }
        self._save_prep()
        return self.prep_data

    def translate_pdf(self) -> Dict:
        if self.prep_data and self.prep_data.get('translated'):
            return self.prep_data
        if not self.prep_data:
            cached = self._load_prep()
            if cached and cached.get('translated'):
                return cached

        extracted = self.extract_pdf()
        lang = extracted['language']
        chunks = extracted['chunks']

        if lang == 'en':
            self.prep_data['chunk_pairs'] = [
                {"original": c, "translated": c} for c in chunks]
            self.prep_data['translated_at'] = str(datetime.now())
        elif lang in self.HELSINKI_MODELS:
            translated_chunks = self._translate_chunks_helsinki(chunks, lang)
            self.prep_data['chunks'] = translated_chunks
            self.prep_data['chunk_pairs'] = [{"original": o, "translated": t}
                                             for o, t in zip(chunks, translated_chunks)]
            self.prep_data['translated'] = True
            self.prep_data['translation_model'] = 'helsinki'
            self.prep_data['translated_at'] = str(datetime.now())
            self._unload_translator()
        else:
            self.prep_data['chunk_pairs'] = [
                {"original": c, "translated": c} for c in chunks]
            self.prep_data['unsupported'] = True
            self.prep_data['translated_at'] = str(datetime.now())

        self._save_prep()
        return self.prep_data

    def load_model(self, model_name: str, task_name: str):
        if task_name in self.models:
            return self.models[task_name]
        if self.models:
            self._clear_gpu_memory()
        self._log(f"  Loading {task_name} model...")
        self.models[task_name] = pipeline("text-classification", model=model_name,
                                          device=self.device, batch_size=self.BATCH_SIZE)
        return self.models[task_name]

    def filter_climate_chunks(self) -> Dict:
        if self.bert_data and self.bert_data.get('filtered'):
            return self.bert_data
        cached = self._load_bert()
        if cached and cached.get('filtered'):
            return cached
        if not self.prep_data:
            self.prep_data = self._load_prep()
            if not self.prep_data:
                raise FileNotFoundError("Run extract_pdf() first")

        chunks = self.prep_data['chunks']
        chunk_ids = self.prep_data.get(
            'chunk_ids', [f"chunk_{i:03d}" for i in range(len(chunks))])
        source = "translated" if self.prep_data.get(
            'translated') else "original"

        self._unload_translator()
        self._clear_gpu_memory()

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector", "detector")
        climate_chunks = []

        for i in range(0, len(chunks), self.BATCH_SIZE):
            batch = chunks[i:i+self.BATCH_SIZE]
            batch_ids = chunk_ids[i:i+self.BATCH_SIZE]
            try:
                results = detector(batch, truncation=True, max_length=512)
                for chunk, chunk_id, result in zip(batch, batch_ids, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'chunk_id': chunk_id, 'company': self.prep_data.get('company'),
                            'company_id': self.prep_data.get('company_id'),
                            'year': self.prep_data.get('year'), 'text': chunk,
                            'detector_score': result['score'], 'detector_label': result['label']
                        })
            except:
                self._clear_gpu_memory()

        kept_pct = len(climate_chunks)/len(chunks)*100 if chunks else 0

        self.bert_data = {
            'pdf_path': str(self.pdf_path), 'company': self.prep_data.get('company'),
            'company_id': self.prep_data.get('company_id'), 'year': self.prep_data.get('year'),
            'language': self.prep_data.get('language', 'unknown'), 'source': source,
            'filtered': True, 'filtered_at': datetime.now().isoformat(),
            'total_chunks': len(chunks), 'climate_chunks': len(climate_chunks),
            'kept_percentage': kept_pct, 'chunks': climate_chunks
        }
        self._save_bert()
        return self.bert_data

    def analyze_specificity(self) -> Dict:
        if self.bert_data and self.bert_data.get('specificity_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data:
            self.filter_climate_chunks()

        chunks = self.bert_data['chunks']
        if 'detector' in self.models:
            del self.models['detector']
            self._clear_gpu_memory()

        specificity = self.load_model(
            "climatebert/distilroberta-base-climate-specificity", "specificity")

        for i in range(0, len(chunks), self.BATCH_SIZE):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = specificity(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['specificity_label'] = result['label']
                    chunk['specificity_score'] = result['score']
            except:
                self._clear_gpu_memory()

        self.bert_data['specificity_analyzed'] = True
        self.bert_data['specificity_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_sentiment(self) -> Dict:
        if self.bert_data and self.bert_data.get('sentiment_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('specificity_analyzed'):
            self.analyze_specificity()

        chunks = self.bert_data['chunks']
        if 'specificity' in self.models:
            del self.models['specificity']
            self._clear_gpu_memory()

        sentiment = self.load_model(
            "climatebert/distilroberta-base-climate-sentiment", "sentiment")

        for i in range(0, len(chunks), self.BATCH_SIZE):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = sentiment(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['sentiment_label'] = result['label']
                    chunk['sentiment_score'] = result['score']
            except:
                self._clear_gpu_memory()

        self.bert_data['sentiment_analyzed'] = True
        self.bert_data['sentiment_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_commitments(self) -> Dict:
        if self.bert_data and self.bert_data.get('commitment_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('sentiment_analyzed'):
            self.analyze_sentiment()

        chunks = self.bert_data['chunks']
        if 'sentiment' in self.models:
            del self.models['sentiment']
            self._clear_gpu_memory()

        commitment = self.load_model(
            "climatebert/distilroberta-base-climate-commitment", "commitment")

        for i in range(0, len(chunks), self.BATCH_SIZE):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = commitment(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['commitment_label'] = result['label']
                    chunk['commitment_score'] = result['score']
            except:
                self._clear_gpu_memory()

        self.bert_data['commitment_analyzed'] = True
        self.bert_data['commitment_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_netzero(self) -> Dict:
        """Analyze which climate chunks are specifically about net-zero/reduction."""
        if self.bert_data and self.bert_data.get('netzero_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('commitment_analyzed'):
            self.analyze_commitments()

        chunks = self.bert_data['chunks']
        if not chunks:
            self.bert_data['netzero_analyzed'] = True
            self.bert_data['netzero_count'] = 0
            self.bert_data['netzero_pct'] = 0
            self._save_bert()
            return self.bert_data

        # Unload previous model
        if 'commitment' in self.models:
            del self.models['commitment']
            self._clear_gpu_memory()

        netzero = self.load_model("climatebert/netzero-reduction", "netzero")

        for i in range(0, len(chunks), self.BATCH_SIZE):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = netzero(batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['netzero_label'] = result['label']
                    chunk['netzero_score'] = result['score']
            except Exception as e:
                self._log(f"  ⚠ Netzero batch error: {e}")
                self._clear_gpu_memory()

        # Summary stats
        n0_chunks = [c for c in chunks if c.get(
            'netzero_label') == 'reduction']

        self.bert_data['netzero_analyzed'] = True
        self.bert_data['netzero_analyzed_at'] = datetime.now().isoformat()
        self.bert_data['netzero_count'] = len(n0_chunks)
        self.bert_data['netzero_pct'] = len(
            n0_chunks) / len(chunks) * 100 if chunks else 0
        self._save_bert()
        return self.bert_data

    def get_final_results(self) -> Optional[Dict]:
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data:
            print(f"❌ No analysis found")
            return None
        required = ['filtered', 'specificity_analyzed',
                    'sentiment_analyzed', 'commitment_analyzed']
        missing = [r for r in required if not self.bert_data.get(r)]
        if missing:
            print(f"⚠ Incomplete: {', '.join(missing)}")
        return self.bert_data

    # === MAIN ENTRY POINT ===

    def process_pdfs(self, path: str, skip_errors: bool = True) -> Dict:
        """
        Process single PDF or directory of PDFs.

        Args:
            path: Path to PDF file or directory containing PDFs
            skip_errors: Continue on errors (default: True)

        Returns:
            Stats dictionary with processing summary
        """
        target = Path(path)
        if not target.exists():
            print(f"❌ Path not found: {path}")
            return None

        if target.is_file():
            if target.suffix.lower() != '.pdf':
                print(f"❌ Not a PDF: {path}")
                return None
            pdf_files = [target]
            root = target.parent
        else:
            pdf_files = sorted(target.rglob("*.pdf"))
            root = target

        if not pdf_files:
            print(f"❌ No PDFs found")
            return None

        n_pdfs = len(pdf_files)

        print(f"\n{'='*60}")
        print(f"CLIMATE REPORT ANALYZER")
        print(f"{'='*60}")
        print(f"  PDFs to process: {n_pdfs}")
        print(f"  Cache directory: {self.cache_dir}")
        print(f"  GPU: {'Yes' if self.device >= 0 else 'No (CPU)'}")
        print(f"{'='*60}\n")

        stats = {
            'total': n_pdfs, 'extracted': 0, 'translated': 0,
            'filtered': 0, 'analyzed': 0, 'skipped': 0, 'errors': [],
            'start_time': time.time()
        }

        # Processing steps for the step bar
        STEPS = ['Extract', 'Translate', 'Filter',
                 'Specificity', 'Sentiment', 'Commitment', 'NetZero']

        # Master progress bar for files (position=0, top)
        pbar_files = tqdm(
            total=n_pdfs,
            desc="Files",
            unit="pdf",
            position=0,
            leave=True,
            dynamic_ncols=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
        )

        # Step progress bar for current file (position=1, below master)
        pbar_steps = tqdm(
            total=len(STEPS),
            desc="Starting...",
            position=1,
            leave=False,
            dynamic_ncols=True,
            bar_format='{desc} [{bar}] {n}/{total}'
        )

        def update_step(step_idx: int, step_name: str, filename: str):
            """Update step bar to show current step."""
            # Truncate filename for display
            if len(filename) > 40:
                filename = "..." + filename[-37:]
            pbar_steps.n = step_idx
            pbar_steps.set_description(f"{filename} → {step_name}")
            pbar_steps.refresh()

        for pdf_file in pdf_files:
            relative_path = pdf_file.relative_to(root)
            display_name = str(relative_path)

            # Reset step bar for new file
            pbar_steps.n = 0
            pbar_steps.refresh()

            try:
                self.set_pdf_path(str(pdf_file))

                # Check if already fully processed
                bert_cache = self._get_cache_path('bert')
                if bert_cache.exists():
                    with open(bert_cache, 'r') as f:
                        cached = json.load(f)
                    if cached.get('netzero_analyzed'):
                        stats['skipped'] += 1
                        stats['extracted'] += 1
                        stats['filtered'] += 1
                        stats['analyzed'] += 1
                        if cached.get('source') == 'translated':
                            stats['translated'] += 1
                        # Show as cached/skipped
                        pbar_steps.n = len(STEPS)
                        pbar_steps.set_description(
                            f"{display_name[-30:]} → Cached ✓")
                        pbar_steps.refresh()
                        pbar_files.update(1)
                        continue

                # Extract
                update_step(1, STEPS[0], display_name)
                self.extract_pdf()
                stats['extracted'] += 1

                # Translate
                update_step(2, STEPS[1], display_name)
                result = self.translate_pdf()
                if result.get('translated'):
                    stats['translated'] += 1

                # Filter
                update_step(3, STEPS[2], display_name)
                self.filter_climate_chunks()
                stats['filtered'] += 1

                # Specificity
                update_step(4, STEPS[3], display_name)
                self.analyze_specificity()

                # Sentiment
                update_step(5, STEPS[4], display_name)
                self.analyze_sentiment()

                # Commitment
                update_step(6, STEPS[5], display_name)
                self.analyze_commitments()

                # NetZero
                update_step(7, STEPS[6], display_name)
                self.analyze_netzero()
                stats['analyzed'] += 1

                # Mark complete
                pbar_steps.n = len(STEPS)
                pbar_steps.set_description(f"{display_name[-30:]} → Done ✓")
                pbar_steps.refresh()

                self._clear_gpu_memory()

            except Exception as e:
                stats['errors'].append(
                    {'file': str(relative_path), 'error': str(e)})
                pbar_steps.set_description(f"{display_name[-30:]} → Error ✗")
                pbar_steps.refresh()
                if not skip_errors:
                    pbar_steps.close()
                    pbar_files.close()
                    raise
                self._emergency_gpu_cleanup()

            pbar_files.update(1)

        pbar_steps.close()
        pbar_files.close()

        stats['elapsed_time'] = time.time() - stats['start_time']
        stats['avg_time_per_pdf'] = stats['elapsed_time'] / \
            n_pdfs if n_pdfs else 0

        # Final summary
        print(f"\n{'='*60}")
        print(f"COMPLETE")
        print(f"{'='*60}")
        print(f"  Processed:   {stats['extracted']}/{stats['total']} PDFs")
        print(f"  From cache:  {stats['skipped']}")
        print(f"  Translated:  {stats['translated']}")
        print(f"  Analyzed:    {stats['analyzed']}")
        print(f"  Errors:      {len(stats['errors'])}")
        print(f"  Total time:  {stats['elapsed_time']/60:.1f} min")
        print(f"  Avg per PDF: {stats['avg_time_per_pdf']:.1f}s")
        print(f"{'='*60}\n")

        if stats['errors']:
            print("Errors encountered:")
            for err in stats['errors'][:5]:
                print(f"  • {err['file']}: {err['error'][:60]}")
            if len(stats['errors']) > 5:
                print(f"  ... and {len(stats['errors'])-5} more")

        return stats


# =============================================================================
# USAGE
# =============================================================================
if __name__ == "__main__":
    analyzer = ClimateReportAnalyzer()

    # Process all PDFs in directory
    stats = analyzer.process_pdfs("../data/reports")

    # Or single file
    # stats = analyzer.process_pdfs("data/reports/SSAB/005_2015_annual_report.pdf")

✓ GPU: NVIDIA T1200 Laptop GPU (3.9GB)

CLIMATE REPORT ANALYZER
  PDFs to process: 204
  Cache directory: cache
  GPU: Yes



Files:   0%|          | 0/204 [00:00<?]

Starting... [          ] 0/7


COMPLETE
  Processed:   204/204 PDFs
  From cache:  204
  Translated:  23
  Analyzed:    204
  Errors:      0
  Total time:  0.0 min
  Avg per PDF: 0.0s

